# SIGMOD Exp 3: Join-Update-Join

Representative derived-state microbenchmark over TPC-H Q14-style inputs. `SNAP` and `IVMH` now rebuild/materialize from a page-based heap MVCC base, while MVHT variants maintain the derived hash state directly. The notebook reports all variants at a fixed `0.01%` update volume.

The benchmark reuses `tpch_q14_join_update_join` and keeps the paper's grouped-stacked-bar style.

In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', 'pandas', 'matplotlib', 'numpy'])
print('done')

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

ROOT = Path('../../').resolve()
sys.path.append(str(ROOT / 'benches'))
import importlib
import sigmod_exp_common as _sigmod_exp_common
importlib.reload(_sigmod_exp_common)

from sigmod_exp_common import (
    TOL,
    SIGMOD_BUCKET_NUM,
    SIGMOD_FIXED_UPDATE_PCT,
    SIGMOD_REPEAT,
    SIGMOD_TPCH_SF,
    SIGMOD_TRIM,
    SIGMOD_WARMUP,
    apply_paper_style,
    current_run_stamp,
    display_name,
    ensure_dirs,
    normalize_repair,
    normalize_table,
    resolve_tpch_file,
    resolve_update_file,
    run_checked,
)

apply_paper_style(ROOT)

EXP_DIR = (ROOT / 'benches' / 'sigmod_exp3_join_update_join').resolve()
DATA_DIR = EXP_DIR / 'data'
FIGS_DIR = EXP_DIR / 'figs'
ensure_dirs(DATA_DIR, FIGS_DIR)

TPCH_DIR = (ROOT / 'benches' / 'sigmod' / 'tpch_data').resolve()
GEN_UPDATES = (ROOT / 'benches' / 'sigmod' / 'generate_updates.py').resolve()
BIN = ROOT / 'target' / 'release' / 'tpch_q14_join_update_join'

SF = SIGMOD_TPCH_SF
DIST = 'uniform'
BUCKET_NUM = SIGMOD_BUCKET_NUM
WARMUP = SIGMOD_WARMUP
REPEAT = SIGMOD_REPEAT
TRIM = SIGMOD_TRIM
FIXED_UPDATE_PCT = SIGMOD_FIXED_UPDATE_PCT
RUN_STAMP = current_run_stamp()

PART_FILE = resolve_tpch_file(TPCH_DIR, 'part', SF)
PROBE_FILE = resolve_tpch_file(TPCH_DIR, 'lineitem_probe', SF, '_1995-09-01_1995-10-01.tbl')

ALL_SERIES = [
    ('naive', 'nr'),
    ('ivmh', 'nr'),
    ('heap', 'nr'), ('heap', 'rr'), ('heap', 'wr'),
    ('chain', 'wr'),
    ('par', 'nr'), ('par', 'rr'), ('par', 'wr'),
]
print('PART :', PART_FILE)
print('PROBE:', PROBE_FILE)
print('BIN  :', BIN)
print('STAMP:', RUN_STAMP)

In [ ]:
print('Building tpch_q14_join_update_join...')
run_checked(['cargo', 'build', '--release', '--bin', 'tpch_q14_join_update_join'], ROOT)
print('Build OK')

In [ ]:
def ensure_update_file(pct):
    try:
        return resolve_update_file(TPCH_DIR, SF, pct, DIST)
    except FileNotFoundError:
        print(f'Generating update file for {pct}%...')
        run_checked([sys.executable, str(GEN_UPDATES), str(PART_FILE), str(pct), str(SF), '--output-dir', str(TPCH_DIR)], ROOT)
        return resolve_update_file(TPCH_DIR, SF, pct, DIST)


def run_q14(table_type, repair_mode, pct, output_csv):
    updates_file = ensure_update_file(pct)
    result = run_checked([
        str(BIN),
        '--part-file', str(PART_FILE),
        '--lineitem-file', str(PROBE_FILE),
        '--updates-file', str(updates_file),
        '--table-type', table_type,
        '--repair-mode', repair_mode,
        '--bucket-num', str(BUCKET_NUM),
        '--warmup', str(WARMUP),
        '--repeat', str(REPEAT),
        '--trim', str(TRIM),
        '--update-pct', str(pct),
        '--distribution', DIST,
        '--output-csv', str(output_csv),
    ], ROOT, quiet=True)
    key_lines = [line for line in result.stdout.splitlines() if 'total_ms=' in line or 'trimmed mean' in line]
    for line in key_lines:
        print(' ', line)

FIXED_CSV = DATA_DIR / f'sigmod_exp3_fixed_{RUN_STAMP}.csv'
if FIXED_CSV.exists():
    FIXED_CSV.unlink()
for table_type, repair_mode in ALL_SERIES:
    print(f'fixed run: {table_type}/{repair_mode}')
    run_q14(table_type, repair_mode, FIXED_UPDATE_PCT, FIXED_CSV)

df_fixed = pd.read_csv(FIXED_CSV)
df_fixed['table'] = df_fixed['table_type'].map(normalize_table)
df_fixed['repair'] = df_fixed['repair_mode'].map(normalize_repair)
display(df_fixed[['table_type', 'repair_mode', 'join1_build_ms', 'join1_probe_ms', 'update_ms', 'join2_build_ms', 'join2_probe_ms', 'total_ms']])

In [ ]:
PHASES = [
    ('join1_build_ms', 'J1 Build', TOL['blue'], True),
    ('join1_probe_ms', 'J1 Probe', TOL['red'], False),
    ('update_ms', 'Update', TOL['yellow'], True),
    ('join2_build_ms', 'J2 Build', TOL['purple'], True),
    ('join2_probe_ms', 'J2 Probe', TOL['green'], False),
]
HATCH = {'J1 Probe': '///', 'J2 Probe': '\\\\\\\\'}
TABLE_ORDER = ['naive', 'ivmh', 'heap', 'chain', 'par']
REPAIR_ORDER = {
    'naive': ['NR'],
    'ivmh': ['NR'],
    'heap': ['NR', 'RR', 'WR'],
    'chain': ['WR'],
    'par': ['NR', 'RR', 'WR'],
}
BASELINES = {'naive', 'ivmh'}

fig, ax = plt.subplots(figsize=(8.6, 5.4))
bar_x, minor_labels, major_centers, major_labels, combos = [], [], [], [], []
x = 0.0
for table in TABLE_ORDER:
    subs = REPAIR_ORDER[table]
    start = x
    for repair in subs:
        combos.append((table, repair))
        bar_x.append(x)
        minor_labels.append('' if table in BASELINES else repair)
        x += 0.78
    major_centers.append((start + (x - 0.78)) / 2.0)
    major_labels.append(display_name(table, ''))
    x += 0.36

for idx, (table, repair) in enumerate(combos):
    row = df_fixed[(df_fixed['table'] == table) & (df_fixed['repair'] == repair)]
    if row.empty:
        continue
    row = row.iloc[0]
    bottom = 0.0
    for col, label, color, is_write in PHASES:
        value = float(row[col])
        if is_write:
            ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color=color, edgecolor='black', linewidth=0.4)
        else:
            ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='white', edgecolor='black', linewidth=0.4)
            ax.bar(bar_x[idx], value, bottom=bottom, width=0.58, color='none', edgecolor=color, linewidth=0.9, hatch=HATCH[label])
        bottom += value

ax.set_xticks([])
ax.set_ylabel('Duration (ms)')
ax.yaxis.grid(True, linestyle='--', linewidth=0.6, alpha=0.6)
for xi, label in zip(bar_x, minor_labels):
    ax.text(xi, -0.04, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=9, clip_on=False)
for xc, label in zip(major_centers, major_labels):
    ax.text(xc, -0.09, label, ha='center', va='top', transform=ax.get_xaxis_transform(), fontsize=10, clip_on=False)
legend_handles = []
legend_labels = []
for col, label, color, is_write in PHASES[::-1]:
    if is_write:
        patch = Patch(facecolor=color, edgecolor='black', linewidth=0.4)
    else:
        patch = Patch(facecolor='white', edgecolor=color, linewidth=0.9, hatch=HATCH[label])
    legend_handles.append(patch)
    legend_labels.append(label)
ax.legend(legend_handles, legend_labels, title='Operations', loc='upper right', framealpha=0.95)
out_pdf = FIGS_DIR / f'sigmod_exp3_all_variants_{RUN_STAMP}.pdf'
fig.tight_layout()
fig.savefig(out_pdf, format='pdf')
plt.show()
print('Saved', out_pdf)

In [ ]:
print('Exp 3 update-volume sweep removed; this notebook now reports the fixed 0.01% point only.')

In [ ]:
# Update-volume sweep figure removed.